# 2. Training, Model Registry, and Lineage

This notebook shows the handoff from a SageMaker training job to an S3 model artifact, a Model Registry version, and lineage inspection. The training script and container image are placeholders so you can connect the workflow to your own model.

In [ ]:
import boto3
import sagemaker
from sagemaker.estimator import Estimator

REGION = "us-east-1"
BUCKET = "replace-with-your-dev-bucket"
ROLE_ARN = "replace-with-sagemaker-execution-role"
IMAGE_URI = "replace-with-training-image"
MODEL_GROUP = "taxi-duration-models-dev"
sm = boto3.client("sagemaker", region_name=REGION)
session = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))

## Submit training

SageMaker uploads the contents of `/opt/ml/model` from the training container to the configured S3 output path. That resulting artifact is the input to model registration.

In [ ]:
estimator = Estimator(
    image_uri=IMAGE_URI,
    role=ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET}/models/",
    sagemaker_session=session,
)
# Replace the placeholder input with your Feature Store/Athena training dataset.
estimator.fit({"training": f"s3://{BUCKET}/training/"}, wait=True)
model_artifact = estimator.model_data
print(model_artifact)

## Register a model version

Keep the model pending until evaluation is complete. In production, metrics and data-quality evidence should be attached before approval.

In [ ]:
response = sm.create_model_package(
    ModelPackageGroupName=MODEL_GROUP,
    ModelApprovalStatus="PendingManualApproval",
    ModelPackageDescription="Development model for learning",
    InferenceSpecification={
        "Containers": [{"Image": IMAGE_URI, "ModelDataUrl": model_artifact}],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
)
model_package_arn = response["ModelPackageArn"]
print(model_package_arn)

## Inspect lineage

SageMaker can associate training inputs, jobs, artifacts, and models in a lineage graph. Use the console or the SDK APIs below to explore the relationships created by your run.

In [ ]:
job_name = estimator.latest_training_job.name
training_job = sm.describe_training_job(TrainingJobName=job_name)
print(training_job["TrainingJobArn"])
print("Open SageMaker Studio lineage or Model Dashboard to inspect the graph.")